<a href="https://colab.research.google.com/github/AllyApitchaya/msc-adr-prediction/blob/main/notebooks/04_gnn_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/MSc_ADR_Project'
print(f"Project path: {PROJECT_PATH}\n")

# Check that a GPU is available. GNN training is far slower on CPU,
# so a GPU runtime is required for this notebook.
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"PyTorch version: {torch.__version__}")
else:
    print("WARNING: No GPU detected.")
    print("Go to Runtime > Change runtime type > T4 GPU, then "
          "re-run this cell.")

Mounted at /content/drive
Project path: /content/drive/MyDrive/MSc_ADR_Project

GPU available: Tesla T4
PyTorch version: 2.10.0+cu128


In [2]:
# RDKit converts SMILES strings into molecular structures, giving
# access to atoms and bonds. It is needed to turn each drug into
# a graph. Colab does not preinstall RDKit.
!pip install rdkit -q
print("RDKit installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 51.3 MB/s eta 0:00:00
RDKit installed.


In [3]:
# PyTorch Geometric (PyG) provides the building blocks for graph
# neural networks. Recent versions can be installed directly with
# pip and will match the existing PyTorch installation.
!pip install torch-geometric -q
print("torch-geometric installed.")

# Verify the installation
import torch_geometric
print(f"PyTorch Geometric version: {torch_geometric.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:00
torch-geometric installed.
PyTorch Geometric version: 2.7.0


In [4]:
import pandas as pd
import os

# Load the model-ready datasets created in notebook 02. The GNN
# uses the SMILES string of each drug to build a molecular graph.
processed_dir = os.path.join(PROJECT_PATH, 'data', 'processed')

train_df = pd.read_csv(os.path.join(processed_dir, 'train.csv'))
val_df   = pd.read_csv(os.path.join(processed_dir, 'val.csv'))
test_df  = pd.read_csv(os.path.join(processed_dir, 'test.csv'))

print(f"Train: {len(train_df):,} pairs")
print(f"Val:   {len(val_df):,} pairs")
print(f"Test:  {len(test_df):,} pairs\n")

# Each unique drug has one SMILES string. Collect them once, since
# building a molecular graph for every row would be wasteful.
drug_smiles = (train_df[['drug', 'smiles']]
               .drop_duplicates()
               .reset_index(drop=True))

print(f"Unique drugs to convert into graphs: {len(drug_smiles)}\n")
print(drug_smiles.to_string(index=False))

Train: 8,240 pairs
Val:   4,912 pairs
Test:  9,016 pairs

Unique drugs to convert into graphs: 15

         drug                                                                           smiles
    glipizide                      CC1=CN=C(C=N1)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC(=O)NC3CCCCC3
  nateglinide                                       CC(C)C1CCC(CC1)C(=O)NC(CC2=CC=CC=C2)C(=O)O
    metformin                                                                CN(C)C(=N)N=C(N)N
    glyburide                 COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC(=O)NC3CCCCC3
dapagliflozin                          CCOC1=CC=C(C=C1)CC2=C(C=CC(=C2)C3C(C(C(C(O3)CO)O)O)O)Cl
  saxagliptin                                  C1C2CC2N(C1C#N)C(=O)C(C34CC5CC(C3)CC(C5)(C4)O)N
     acarbose CC1C(C(C(C(O1)OC2C(OC(C(C2O)O)OC(C(CO)O)C(C(C=O)O)O)CO)O)O)NC3C=C(C(C(C3O)O)O)CO
  repaglinide                      CCOC1=C(C=CC(=C1)CC(=O)NC(CC(C)C)C2=CC=CC=C2N3CCCCC3)C(=O)O
rosiglitazone                                 

In [5]:
from rdkit import Chem
import torch
from torch_geometric.data import Data


def atom_features(atom):
    """Return a feature vector for a single atom."""
    return [
        atom.GetAtomicNum(),        # element (C=6, N=7, O=8, ...)
        atom.GetDegree(),           # number of bonded neighbours
        atom.GetFormalCharge(),     # formal charge
        atom.GetTotalNumHs(),       # attached hydrogens
        int(atom.GetIsAromatic()),  # 1 if aromatic, else 0
    ]


def smiles_to_graph(smiles):
    """Convert a SMILES string into a PyG graph (Data object).
    Nodes are atoms; edges are chemical bonds.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # SMILES could not be parsed

    # Node feature matrix: one row per atom
    node_feats = [atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(node_feats, dtype=torch.float)

    # Edge list: each bond becomes two directed edges (i->j, j->i)
    edge_list = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_list += [[i, j], [j, i]]
    edge_index = torch.tensor(edge_list, dtype=torch.long).t()

    return Data(x=x, edge_index=edge_index)


# Test the function on a single, simple molecule first
test_smiles = drug_smiles.loc[
    drug_smiles['drug'] == 'metformin', 'smiles'].iloc[0]
g = smiles_to_graph(test_smiles)

print(f"Test molecule: metformin")
print(f"SMILES: {test_smiles}\n")
print(f"Number of atoms (nodes): {g.num_nodes}")
print(f"Number of bonds (edges): {g.num_edges // 2}")
print(f"Node feature shape: {g.x.shape}  "
      f"(atoms x features)\n")
print("Node features (each row = one atom):")
print(g.x)

Test molecule: metformin
SMILES: CN(C)C(=N)N=C(N)N

Number of atoms (nodes): 9
Number of bonds (edges): 8
Node feature shape: torch.Size([9, 5])  (atoms x features)

Node features (each row = one atom):
tensor([[6., 1., 0., 3., 0.],
        [7., 3., 0., 0., 0.],
        [6., 1., 0., 3., 0.],
        [6., 3., 0., 0., 0.],
        [7., 1., 0., 1., 0.],
        [7., 2., 0., 0., 0.],
        [6., 3., 0., 0., 0.],
        [7., 1., 0., 2., 0.],
        [7., 1., 0., 2., 0.]])


In [6]:
# Build a molecular graph for every drug and store them in a
# dictionary keyed by drug name. The GNN will look up a drug's
# graph by name when processing each drug-ADR pair.
drug_graphs = {}

print("Building molecular graphs for all drugs:\n")
for _, row in drug_smiles.iterrows():
    graph = smiles_to_graph(row['smiles'])
    if graph is None:
        print(f"  WARNING: failed to parse {row['drug']}")
        continue
    drug_graphs[row['drug']] = graph
    print(f"  {row['drug']:16s} "
          f"{graph.num_nodes:3d} atoms, "
          f"{graph.num_edges // 2:3d} bonds")

print(f"\nGraphs built for {len(drug_graphs)} / "
      f"{len(drug_smiles)} drugs")

# The node feature dimension is needed later when defining the
# GNN's input layer
NODE_FEAT_DIM = next(iter(drug_graphs.values())).x.shape[1]
print(f"Node feature dimension: {NODE_FEAT_DIM}")

Building molecular graphs for all drugs:

  glipizide         31 atoms,  33 bonds
  nateglinide       23 atoms,  24 bonds
  metformin          9 atoms,   8 bonds
  glyburide         33 atoms,  35 bonds
  dapagliflozin     28 atoms,  30 bonds
  saxagliptin       23 atoms,  27 bonds
  acarbose          44 atoms,  46 bonds
  repaglinide       33 atoms,  35 bonds
  rosiglitazone     25 atoms,  27 bonds
  sitagliptin       28 atoms,  30 bonds
  pioglitazone      25 atoms,  27 bonds
  canagliflozin     31 atoms,  34 bonds
  linagliptin       35 atoms,  39 bonds
  empagliflozin     31 atoms,  34 bonds
  glimepiride       34 atoms,  36 bonds

Graphs built for 15 / 15 drugs
Node feature dimension: 5


In [7]:
# ADR terms are text and must be mapped to integer indices before
# the model can use them. Build a vocabulary from all ADRs that
# appear across the train, validation and test splits.
all_adrs = pd.concat([train_df['adr'],
                      val_df['adr'],
                      test_df['adr']]).unique()

adr_to_idx = {adr: idx for idx, adr in enumerate(sorted(all_adrs))}
NUM_ADRS = len(adr_to_idx)

print(f"ADR vocabulary size: {NUM_ADRS:,} unique ADRs\n")

# Show a few entries
print("Sample ADR -> index mapping:")
for adr in list(adr_to_idx)[:5]:
    print(f"  {adr:35s} -> {adr_to_idx[adr]}")

# Quick sanity check on a known ADR
known = 'Lactic acidosis'
if known in adr_to_idx:
    print(f"\n'{known}' is index {adr_to_idx[known]}")

ADR vocabulary size: 3,576 unique ADRs

Sample ADR -> index mapping:
  5-hydroxyindolacetic acid in urine increased -> 0
  ABO incompatibility                 -> 1
  Abdominal abscess                   -> 2
  Abdominal adhesions                 -> 3
  Abdominal compartment syndrome      -> 4

'Lactic acidosis' is index 1902


In [8]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool


class ADRPredictor(nn.Module):
    """Predicts whether a drug-ADR pair is a true association.

    The drug is encoded from its molecular graph with a Graph
    Attention Network (GAT). The ADR is encoded with a learnable
    embedding. The two encodings are concatenated and passed
    through a small classifier.
    """

    def __init__(self, node_feat_dim, num_adrs,
                 hidden_dim=64, embed_dim=64):
        super().__init__()

        # --- Drug branch: GAT over the molecular graph ---
        self.gat1 = GATConv(node_feat_dim, hidden_dim, heads=4)
        # GATConv with 4 heads outputs hidden_dim * 4 features
        self.gat2 = GATConv(hidden_dim * 4, hidden_dim, heads=1)

        # --- ADR branch: a learnable embedding per ADR ---
        self.adr_embed = nn.Embedding(num_adrs, embed_dim)

        # --- Classifier: combines drug and ADR encodings ---
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim + embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, graph_batch, adr_idx):
        # Drug branch
        x, edge_index = graph_batch.x, graph_batch.edge_index
        x = F.elu(self.gat1(x, edge_index))
        x = F.elu(self.gat2(x, edge_index))
        # Pool atom features into one vector per molecule
        drug_vec = global_mean_pool(x, graph_batch.batch)

        # ADR branch
        adr_vec = self.adr_embed(adr_idx)

        # Combine and classify
        combined = torch.cat([drug_vec, adr_vec], dim=1)
        logit = self.classifier(combined).squeeze(-1)
        return logit


# Instantiate the model and move it to the GPU
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'cpu')
model = ADRPredictor(NODE_FEAT_DIM, NUM_ADRS).to(device)

print(f"Model created on: {device}\n")
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {n_params:,}")

Model created on: cuda

ADRPredictor(
  (gat1): GATConv(5, 64, heads=4)
  (gat2): GATConv(256, 64, heads=1)
  (adr_embed): Embedding(3576, 64)
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)

Total trainable parameters: 255,809


In [9]:
from torch_geometric.data import Dataset, Batch
from torch.utils.data import DataLoader

# A custom dataset: each item is one drug-ADR pair, represented
# by the drug's molecular graph, the ADR index, and the label.
class ADRDataset(Dataset):
    def __init__(self, df, drug_graphs, adr_to_idx):
        super().__init__()
        self.rows = df.reset_index(drop=True)
        self.drug_graphs = drug_graphs
        self.adr_to_idx = adr_to_idx

    def len(self):
        return len(self.rows)

    def get(self, idx):
        row = self.rows.iloc[idx]
        graph = self.drug_graphs[row['drug']]
        adr_idx = self.adr_to_idx[row['adr']]
        label = float(row['label'])
        return graph, adr_idx, label


def collate(batch):
    """Combine a list of (graph, adr_idx, label) into batched
    tensors that the model can process in one forward pass.
    """
    graphs, adr_idxs, labels = zip(*batch)
    graph_batch = Batch.from_data_list(list(graphs))
    adr_tensor = torch.tensor(adr_idxs, dtype=torch.long)
    label_tensor = torch.tensor(labels, dtype=torch.float)
    return graph_batch, adr_tensor, label_tensor


BATCH_SIZE = 32

train_ds = ADRDataset(train_df, drug_graphs, adr_to_idx)
val_ds   = ADRDataset(val_df,   drug_graphs, adr_to_idx)
test_ds  = ADRDataset(test_df,  drug_graphs, adr_to_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate)

print(f"Batch size: {BATCH_SIZE}\n")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Inspect one batch to confirm shapes
graph_batch, adr_tensor, label_tensor = next(iter(train_loader))
print(f"\nOne batch:")
print(f"  graph batch: {graph_batch.num_graphs} molecules, "
      f"{graph_batch.x.shape[0]} total atoms")
print(f"  adr indices: {adr_tensor.shape}")
print(f"  labels:      {label_tensor.shape}")

Batch size: 32

Train batches: 258
Val batches:   154
Test batches:  282

One batch:
  graph batch: 32 molecules, 776 total atoms
  adr indices: torch.Size([32])
  labels:      torch.Size([32])


In [10]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Binary classification loss. The model outputs a raw logit, so
# BCEWithLogitsLoss applies the sigmoid internally.
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def train_epoch(model, loader):
    """Train the model for one pass over the data."""
    model.train()
    total_loss = 0.0
    for graph_batch, adr_idx, labels in loader:
        graph_batch = graph_batch.to(device)
        adr_idx = adr_idx.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(graph_batch, adr_idx)
        loss = criterion(logits, labels)
        loss.backward()        # compute gradients
        optimizer.step()       # update parameters

        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate the model without updating parameters."""
    model.eval()
    all_probs, all_labels = [], []
    for graph_batch, adr_idx, labels in loader:
        graph_batch = graph_batch.to(device)
        adr_idx = adr_idx.to(device)

        logits = model(graph_batch, adr_idx)
        probs = torch.sigmoid(logits).cpu()

        all_probs.append(probs)
        all_labels.append(labels)

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()

    return {
        'AUROC': roc_auc_score(labels, probs),
        'AUPRC': average_precision_score(labels, probs),
    }


print("Training and evaluation functions defined.")
print(f"Loss function: BCEWithLogitsLoss")
print(f"Optimizer: Adam (learning rate = 1e-3)")

Training and evaluation functions defined.
Loss function: BCEWithLogitsLoss
Optimizer: Adam (learning rate = 1e-3)


In [11]:
import copy

# Train the GNN. After each epoch, evaluate on the validation set
# and keep a snapshot of the best-performing model. This guards
# against overfitting in later epochs.
NUM_EPOCHS = 20

best_val_auroc = 0.0
best_model_state = None

print(f"Training for {NUM_EPOCHS} epochs...\n")
print(f"{'Epoch':>6} {'Train Loss':>12} "
      f"{'Val AUROC':>11} {'Val AUPRC':>11}")
print("-" * 42)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader)

    # Keep the best model based on validation AUROC
    if val_metrics['AUROC'] > best_val_auroc:
        best_val_auroc = val_metrics['AUROC']
        best_model_state = copy.deepcopy(model.state_dict())
        marker = "  <- best"
    else:
        marker = ""

    print(f"{epoch:>6} {train_loss:>12.4f} "
          f"{val_metrics['AUROC']:>11.4f} "
          f"{val_metrics['AUPRC']:>11.4f}{marker}")

print("-" * 42)
print(f"\nBest validation AUROC: {best_val_auroc:.4f}")

# Restore the best model for final evaluation
model.load_state_dict(best_model_state)
print("Best model restored.")

Training for 20 epochs...

 Epoch   Train Loss   Val AUROC   Val AUPRC
------------------------------------------
     1       0.5748      0.7962      0.8175  <- best
     2       0.4867      0.8274      0.8351  <- best
     3       0.4600      0.8384      0.8436  <- best
     4       0.4240      0.8514      0.8519  <- best
     5       0.3863      0.8592      0.8575  <- best
     6       0.3588      0.8601      0.8551  <- best
     7       0.3289      0.8643      0.8597  <- best
     8       0.3007      0.8695      0.8632  <- best
     9       0.2816      0.8675      0.8598
    10       0.2624      0.8667      0.8580
    11       0.2448      0.8709      0.8618  <- best
    12       0.2240      0.8688      0.8593
    13       0.2106      0.8658      0.8579
    14       0.2008      0.8673      0.8578
    15       0.1855      0.8673      0.8578
    16       0.1773      0.8642      0.8548
    17       0.1628      0.8665      0.8567
    18       0.1552      0.8618      0.8536
    19       

In [12]:
import os

# Final evaluation on the held-out test set. The test set was
# never seen during training or model selection, so this gives an
# unbiased estimate of performance and a fair comparison with the
# baseline models from notebook 03.
test_metrics = evaluate(model, test_loader)

print("=" * 50)
print("GNN MODEL - FINAL TEST SET RESULTS")
print("=" * 50)
print(f"\n  Test AUROC: {test_metrics['AUROC']:.4f}")
print(f"  Test AUPRC: {test_metrics['AUPRC']:.4f}")

# Compare against the XGBoost baseline from notebook 03
BASELINE_AUROC = 0.7008  # from notebook 03

print(f"\n{'-' * 50}")
print("COMPARISON WITH BASELINE")
print(f"{'-' * 50}")
print(f"  XGBoost baseline AUROC: {BASELINE_AUROC:.4f}")
print(f"  GNN model AUROC:        {test_metrics['AUROC']:.4f}")

diff = test_metrics['AUROC'] - BASELINE_AUROC
if diff > 0:
    print(f"\n  GNN improves over baseline by {diff:+.4f} AUROC.")
else:
    print(f"\n  GNN does not exceed baseline ({diff:+.4f}).")

# Save the trained model and its test results
results_dir = os.path.join(PROJECT_PATH, 'results')
models_dir = os.path.join(PROJECT_PATH, 'models')
os.makedirs(models_dir, exist_ok=True)

torch.save(model.state_dict(),
           os.path.join(models_dir, 'gnn_model.pt'))
pd.DataFrame([test_metrics]).to_csv(
    os.path.join(results_dir, 'gnn_test_metrics.csv'), index=False)

print(f"\nSaved model     -> models/gnn_model.pt")
print(f"Saved metrics   -> results/gnn_test_metrics.csv")

GNN MODEL - FINAL TEST SET RESULTS

  Test AUROC: 0.8185
  Test AUPRC: 0.8315

--------------------------------------------------
COMPARISON WITH BASELINE
--------------------------------------------------
  XGBoost baseline AUROC: 0.7008
  GNN model AUROC:        0.8185

  GNN improves over baseline by +0.1177 AUROC.

Saved model     -> models/gnn_model.pt
Saved metrics   -> results/gnn_test_metrics.csv
